# 🎙️ PrecisionVoice - Vietnamese Speech-to-Text

Notebook đơn giản để transcribe audio tiếng Việt sử dụng **faster-whisper** và **Gradio UI**.

### Hướng dẫn
1. **Chọn GPU**: `Runtime` → `Change runtime type` → **T4 GPU**
2. **Chạy từng cell** theo thứ tự từ trên xuống
3. **Sử dụng Gradio link** ở cell cuối để truy cập UI

In [1]:
# @title 1. 🔍 Kiểm tra GPU
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Detected: {gpu_name}")
    print(f"   VRAM: {gpu_mem:.1f} GB")
else:
    print("⚠️ KHÔNG TÌM THẤY GPU!")
    print("👉 Vào Runtime → Change runtime type → T4 GPU")

✅ GPU Detected: Tesla T4
   VRAM: 15.8 GB


In [2]:
# @title 2. 📦 Cài đặt Dependencies
print("Installing dependencies...")
!pip install -q faster-whisper gradio speechbrain scikit-learn librosa nest_asyncio
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✅ Dependencies installed successfully!")

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 21.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 21.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 111.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.9 MB/s eta 0:00:00
✅ Dependencies installed successfully!


In [3]:
# @title 3. 🤖 Load Models (Whisper & SpeechBrain)
import torchaudio
import sys

# 1. Monkeypatch torchaudio BEFORE any speechbrain import
if not hasattr(torchaudio, 'list_audio_backends'):
    torchaudio.list_audio_backends = lambda: []

# 2. Handle SpeechBrain initialization quirks
try:
    import speechbrain.utils.quirks
    import speechbrain.utils.profiling
except (ImportError, AttributeError):
    pass

from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier
import torch
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading EraX-WoW-Turbo model (optimized for Vietnamese)...")
start = time.time()
model = WhisperModel(
    "erax-ai/EraX-WoW-Turbo-V1.1-CT2",
    device=device,
    compute_type="float16" if device == "cuda" else "int8"
)
print(f"✅ Whisper loaded in {time.time() - start:.1f}s")

print("Loading SpeechBrain Speaker Recognition model...")
start = time.time()
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": device}
)
print(f"✅ SpeechBrain loaded in {time.time() - start:.1f}s")

Loading EraX-WoW-Turbo model (optimized for Vietnamese)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded in 28.5s


In [ ]:
# @title 4. 🎤 Khởi chạy Gradio UI
import gradio as gr
import time
import torch
import numpy as np
import librosa
import nest_asyncio
from sklearn.cluster import AgglomerativeClustering, SpectralClustering

# Apply nest_asyncio
nest_asyncio.apply()

def diarize_segments(audio_path, segments, num_speakers, auto_diarize=True, p=None):
    """Assign speaker labels to Whisper segments using SpeechBrain embeddings."""
    if not segments:
        return []
    
    if p:
        p(0.7, desc="2/3: Đang trích xuất đặc trưng giọng nói...")
    
    audio, sr = librosa.load(audio_path, sr=16000)
    embeddings = []
    valid_segments = []
    total_segs = len(segments)
    
    for i, segment in enumerate(segments):
        start_sample = int(segment.start * sr)
        end_sample = int(segment.end * sr)
        if end_sample - start_sample < 160: continue
            
        seg_audio = audio[start_sample:end_sample]
        with torch.no_grad():
            seg_tensor = torch.tensor(seg_audio).unsqueeze(0)
            emb = classifier.encode_batch(seg_tensor)
            embeddings.append(emb.squeeze().cpu().numpy())
            valid_segments.append(segment)
            
        if p and i % 5 == 0:
             p(0.7 + (i/total_segs) * 0.2, desc=f"2/3: Đang xử lý đoạn {i}/{total_segs}...")
    
    if not embeddings or len(embeddings) < 2:
        return [{'start': seg.start, 'end': seg.end, 'text': seg.text, 'speaker': 'Speaker 0'} for seg in segments]

    embeddings = np.array(embeddings)
    if p:
        p(0.9, desc="2/3: Đang phân cụm người nói...")
    
    if auto_diarize:
        clustering = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average', distance_threshold=0.55)
        labels = clustering.fit_predict(embeddings)
    else:
        clustering = SpectralClustering(n_clusters=num_speakers, affinity='cosine', random_state=42)
        labels = clustering.fit_predict(embeddings)
    
    return [{'start': s.start, 'end': s.end, 'text': s.text, 'speaker': f"Speaker {labels[i]}"} for i, s in enumerate(valid_segments)]

def transcribe(audio_path, language, beam_size, vad_filter, auto_diarize, num_speakers, p=gr.Progress()):
    """Transcribe audio file with immediate UI feedback."""
    if audio_path is None:
        yield "⚠️ Vui lòng upload hoặc ghi âm audio!"
        return
    
    # Yield immediately to force UI update
    yield "🚀 Đang khởi tạo và chuẩn bị dữ liệu..."
    p(0, desc="🚀 Đang khởi tạo...")
    
    start_time = time.time()
    
    # 1. Transcribe
    p(0.1, desc="1/3: Đang bắt đầu Whisper transcription...")
    segments_gen, info = model.transcribe(
        audio_path, language=language if language != "auto" else None,
        beam_size=beam_size, vad_filter=vad_filter, word_timestamps=False,
    )
    
    segments = []
    for segment in segments_gen:
        segments.append(segment)
        p(0.1 + (len(segments) % 100) / 1000, desc=f"1/3: Đang nhận diện văn bản ({len(segments)} câu)...")
        # Periodically yield to keep UI alive if needed, but here we just update progress
    
    # 2. Diarize
    p(0.6, desc="2/3: Đang bắt đầu nhận diện người nói...")
    diarized_segments = diarize_segments(audio_path, segments, num_speakers, auto_diarize, p)
    
    # 3. Format output
    p(0.95, desc="3/3: Đang tổng hợp kết quả...")
    result_lines = []
    full_text = []
    detected_speakers = len(set(seg.get('speaker') for seg in diarized_segments))
    
    for seg in diarized_segments:
        speaker = seg.get('speaker', 'Unknown')
        result_lines.append(f"[{seg['start']:.2f}s → {seg['end']:.2f}s] **{speaker}**: {seg['text']}")
        full_text.append(f"{speaker}: {seg['text'].strip()}")
    
    elapsed = time.time() - start_time
    output = f"📊 Ngôn ngữ: {info.language} ({info.language_probability:.1%})\n"
    output += f"👥 Số người nói: {detected_speakers}\n"
    output += f"⏱️ Thời gian xử lý: {elapsed:.1f}s\n"
    output += f"━" * 50 + "\n\n"
    output += "\n".join(result_lines)
    output += f"\n\n" + "━" * 50 + "\n"
    output += f"📝 Transcript đầy đủ:\n{'\n'.join(full_text)}"
    
    yield output

with gr.Blocks(title="PrecisionVoice", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ PrecisionVoice - Vietnamese STT (Diarization supported)")
    gr.Markdown("Upload audio hoặc ghi âm để bắt đầu chuyển đổi và phân biệt người nói.")
    
    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="🔊 Audio Input")
            
            with gr.Row():
                language = gr.Dropdown(choices=["auto", "vi", "en", "zh", "ja", "ko"], value="vi", label="🌐 Ngôn ngữ")
            
            with gr.Group():
                auto_diarize = gr.Checkbox(value=True, label="🔄 Tự động phát hiện số người nói")
                num_speakers = gr.Slider(minimum=1, maximum=10, value=2, step=1, label="👥 Số người nói (thủ công)", interactive=False)
            
            auto_diarize.change(fn=lambda x: gr.update(interactive=not x), inputs=auto_diarize, outputs=num_speakers)
            
            with gr.Accordion("Cài đặt nâng cao", open=False):
                beam_size = gr.Slider(minimum=1, maximum=10, value=5, step=1, label="🎯 Beam Size")
                vad_filter = gr.Checkbox(value=True, label="🔇 VAD Filter")
            
            transcribe_btn = gr.Button("▶️ Bắt đầu xử lý", variant="primary")
        
        with gr.Column(scale=2):
            output_text = gr.Markdown(label="📝 Kết quả Transcription & Diarization")
    
    transcribe_btn.click(
        fn=transcribe,
        inputs=[audio_input, language, beam_size, vad_filter, auto_diarize, num_speakers],
        outputs=output_text
    )
    
    gr.Markdown("---")
    gr.Markdown("*EraX-WoW-Turbo & SpeechBrain ECAPA-TDNN | Powered by faster-whisper*")

import os
is_colab = "COLAB_GPU" in os.environ or "google.colab" in str(get_ipython())
if is_colab:
    demo.queue().launch(share=True, debug=True, show_error=True)
else:
    demo.launch(share=False)

/tmp/ipython-input-130105980.py:42: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="PrecisionVoice", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://76b49811a61ce08b19.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
